# Neural Network Test Run (4) — Plain PyTorch version
Same model and data as the Lightning notebook, but with an explicit training loop (no `lightning` dependency).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split

## Load and prepare data (unchanged)

In [2]:
df = pd.read_table('./iris.txt', sep=",", header=None)
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]

input_values = df[['petal_width', 'sepal_width']]
classes_as_numbers = df['class'].factorize()[0]

input_train, input_test, label_train, label_test = train_test_split(
    input_values, classes_as_numbers,
    test_size=0.25, stratify=classes_as_numbers, random_state=42)

# One-hot labels, as in the original notebook
one_hot_label_train = F.one_hot(torch.tensor(label_train)).type(torch.float32)

# Min-max normalize using TRAIN statistics only
min_vals = input_train.min()
max_vals = input_train.max()
input_train = (input_train - min_vals) / (max_vals - min_vals)
input_test = (input_test - min_vals) / (max_vals - min_vals)

input_train_tensors = torch.tensor(input_train.values).type(torch.float32)
input_test_tensors = torch.tensor(input_test.values).type(torch.float32)

train_dataset = TensorDataset(input_train_tensors, one_hot_label_train)
train_dataloader = DataLoader(train_dataset)  # batch_size=1, no shuffle, same as original
# Optional: DataLoader(train_dataset, batch_size=16, shuffle=True)

## Model
Plain `nn.Module`: `__init__` and `forward` only. The loss, optimizer, and training step now live outside the class.

In [3]:
class MultipleInsOuts(nn.Module):

    def __init__(self):
        super().__init__()
        self.input_to_hidden = nn.Linear(in_features=2, out_features=2, bias=True)
        self.hidden_to_output = nn.Linear(in_features=2, out_features=3, bias=True)

    def forward(self, x):
        hidden = self.input_to_hidden(x)
        return self.hidden_to_output(torch.relu(hidden))  # raw logits; no softmax

torch.manual_seed(42)  # replaces L.seed_everything(42)
model = MultipleInsOuts()
loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

## Training loop and evaluation
This replaces `training_step`, `configure_optimizers`, and `Trainer.fit`.

In [4]:
def train(model, dataloader, loss_fn, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for inputs, labels in dataloader:
            optimizer.zero_grad()              # clear old gradients
            outputs = model(inputs)            # forward pass
            loss = loss_fn(outputs, labels)    # compute loss
            loss.backward()                    # backpropagation
            optimizer.step()                   # update weights
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}: avg loss = {total_loss / len(dataloader):.4f}")

def accuracy(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        predicted = torch.argmax(model(inputs), dim=1)
    return (predicted == torch.tensor(labels)).float().mean().item()

In [5]:
train(model, train_dataloader, loss_fn, optimizer, epochs=10)
print("Test accuracy after 10 epochs:", accuracy(model, input_test_tensors, label_test))

Epoch 1: avg loss = 1.0385
Epoch 10: avg loss = 0.7897
Test accuracy after 10 epochs: 0.6578947305679321


Continue training for 90 more epochs (100 total). The same `model` and `optimizer` objects keep their state, so no checkpoint is needed.

In [6]:
train(model, train_dataloader, loss_fn, optimizer, epochs=90)
print("Test accuracy after 100 epochs:", accuracy(model, input_test_tensors, label_test))

Epoch 1: avg loss = 0.7595
Epoch 10: avg loss = 0.5544
Epoch 20: avg loss = 0.4438
Epoch 30: avg loss = 0.3676
Epoch 40: avg loss = 0.3104
Epoch 50: avg loss = 0.2660
Epoch 60: avg loss = 0.2315
Epoch 70: avg loss = 0.2045
Epoch 80: avg loss = 0.1831
Epoch 90: avg loss = 0.1662
Test accuracy after 100 epochs: 0.9473684430122375


In [7]:
for name, param in model.named_parameters():
    print(name, torch.round(param.data, decimals=2))

input_to_hidden.weight tensor([[ 3.5500,  0.2500],
        [-1.9600,  1.4800]])
input_to_hidden.bias tensor([-0.3400,  1.2400])
hidden_to_output.weight tensor([[-4.2000,  3.1500],
        [ 0.1600,  0.3300],
        [ 1.8700, -3.2900]])
hidden_to_output.bias tensor([ 0.4600,  0.9600, -0.6100])


## Predict a single flower

In [8]:
normalized_values = ([0.2, 3.0] - min_vals) / (max_vals - min_vals)
input_tensor = torch.tensor(normalized_values.values, dtype=torch.float32)

model.eval()
with torch.no_grad():
    print(torch.argmax(model(input_tensor)))

tensor(0)
